In [ ]:
# Cell 0 — install + imports
!pip install -q vllm datasets transformers peft anthropic boto3

import json
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct"
TRAIN_JSONL = "/content/drive/MyDrive/sft/train_dataset_clean.jsonl"  # for repo-overlap

# Auto-detect v4 merged model: prefer local /content/ (atomic, fast),
# fall back to Drive (per the v4 ship "save local first then cp to Drive" lesson)
def _shards_ok(p):
    p = Path(p)
    if not p.exists():
        return False
    shards = list(p.glob("*.safetensors"))
    return len(shards) > 0 and all(s.stat().st_size > 1e6 for s in shards)

V4_CANDIDATES = [
    "/content/sft-v4-merged-for-eval",
    "/content/drive/MyDrive/sft/sft-v4-merged-for-eval",
]
V4_MODEL = next((p for p in V4_CANDIDATES if _shards_ok(p)), None)
assert V4_MODEL, f"No valid v4 merged model found in any of {V4_CANDIDATES} — check shard sizes"
print(f"[ood_eval] using v4_model = {V4_MODEL}")

# Sanity check AWS credentials for AnthropicBedrock (used by pairwise judge)
import boto3
try:
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    print(f"[ood_eval] AWS identity: {identity['Arn']}")
except Exception as e:
    print(f"[ood_eval] WARNING: AWS credentials not configured ({e}); pairwise will fail")

In [ ]:
# Cell 1 — download SWE-CARE, filter overlap
!python swecare_loader.py --train-jsonl {TRAIN_JSONL} --output ood_input.jsonl

!if [ -s ood_input.jsonl ]; then head -1 ood_input.jsonl | python -c "import json, sys; print(json.dumps(json.loads(sys.stdin.read()), indent=2)[:500])"; else echo "(ood_input.jsonl is empty — check repo-overlap filter or --dry-run setting)"; fi
!wc -l ood_input.jsonl

In [ ]:
# Cell 2 — v4 + base inference (~6h sequential on A100 80GB)
!python run_ood_eval.py \
    --input ood_input.jsonl \
    --output ood_preds.jsonl \
    --v4-model {V4_MODEL} \
    --base-model {BASE_MODEL}

!wc -l ood_preds.jsonl

In [ ]:
# Cell 3 — compute all 6 metrics + 3-vote pairwise
!python ood_metrics.py \
    --preds ood_preds.jsonl \
    --labels ood_input.jsonl \
    --output ood_eval_results.json

import json
results = json.load(open('ood_eval_results.json'))
for k, v in results.items():
    if isinstance(v, dict):
        print(f"{k}:")
        for kk, vv in v.items():
            print(f"  {kk}: {vv:.3f}" if isinstance(vv, float) else f"  {kk}: {vv}")
    else:
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
# Cell 4 — visualizations
import json
import matplotlib.pyplot as plt

results = json.load(open('ood_eval_results.json'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: per-difficulty
diffs = results['iou_lenient_by_difficulty']
axes[0].bar(diffs.keys(), diffs.values())
axes[0].set_title('IoU (lenient) by difficulty')
axes[0].set_ylim(0, 1)

# Plot 2: per-problem-domain
doms = results['iou_lenient_by_problem_domain']
axes[1].barh(list(doms.keys()), list(doms.values()))
axes[1].set_title('IoU (lenient) by problem_domain')
axes[1].set_xlim(0, 1)

# Plot 3: aggregate vs ID baseline (v4 ID was 0.18 ROUGE-L; not directly comparable
# but plot the headline OOD numbers)
metrics = ['iou_strict_mean', 'iou_lenient_mean', 'hit_rate_mean', 'hallucination_rate_mean']
axes[2].bar(metrics, [results.get(m, 0) for m in metrics])
axes[2].set_title('Aggregate metrics')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('ood_eval_plots.png', dpi=120)
plt.show()

In [ ]:
# Cell 5 — decision-gate readout per spec
import json
r = json.load(open('ood_eval_results.json'))

print("=" * 60)
print("PHASE 1 DECISION GATE")
print("=" * 60)

pairwise = r.get('pairwise', {}).get('win_rate', 0) * 100
ci_lo = r.get('pairwise', {}).get('win_rate_ci_lo', 0) * 100
ci_hi = r.get('pairwise', {}).get('win_rate_ci_hi', 0) * 100
iou_strict = r['iou_strict_mean'] * 100
iou_lenient = r['iou_lenient_mean'] * 100
hit_lenient = r['hit_rate_mean'] * 100
hit_strict = r.get('hit_rate_strict_mean', 0) * 100
halluc = r['hallucination_rate_mean'] * 100

# Reference points from the ID eval (v4 ship report)
BASE_HALLUC_ID = 14.5  # base model hallucination rate, in-distribution
V4_HALLUC_ID = 7.0     # v4 hallucination rate, in-distribution

print(f"v4 OOD pairwise win:    {pairwise:.1f}%  [CI {ci_lo:.1f}%–{ci_hi:.1f}%]")
print(f"v4 OOD IoU (strict):    {iou_strict:.1f}%")
print(f"v4 OOD IoU (lenient):   {iou_lenient:.1f}%")
print(f"v4 OOD hit-rate (str):  {hit_strict:.1f}%")
print(f"v4 OOD hit-rate (len):  {hit_lenient:.1f}%")
print(f"v4 OOD hallucination:   {halluc:.1f}%  (ID baseline: v4={V4_HALLUC_ID}%, base={BASE_HALLUC_ID}%)")
print()

# Per-bucket dispersion — if max-min > 30 pts, that's "concentrated weakness"
domain_vals = list(r.get('iou_lenient_by_problem_domain', {}).values())
diff_vals = list(r.get('iou_lenient_by_difficulty', {}).values())
domain_spread = (max(domain_vals) - min(domain_vals)) * 100 if domain_vals else 0
diff_spread = (max(diff_vals) - min(diff_vals)) * 100 if diff_vals else 0

print(f"per-domain IoU spread:    {domain_spread:.1f} pts")
print(f"per-difficulty IoU spread: {diff_spread:.1f} pts")
print()

# Hallucination tail check: does OOD hallucination exceed BASE's ID hallucination?
hallucination_regression = halluc > BASE_HALLUC_ID
if hallucination_regression:
    print(f"⚠ HALLUCINATION REGRESSION — OOD halluc ({halluc:.1f}%) > base ID baseline ({BASE_HALLUC_ID}%)")
    print()

# Uneven per-repo signal — domain OR difficulty bucket > 30pt spread
uneven_breakdown = domain_spread >= 30 or diff_spread >= 30

# Spec decision-gate
if pairwise >= 65 and not uneven_breakdown and not hallucination_regression:
    branch = "2C — ship v4 as-is, build OOD-aware inference hardening"
elif pairwise >= 65 and uneven_breakdown:
    weakest_bucket = (
        min(r.get('iou_lenient_by_problem_domain', {}).items(), key=lambda kv: kv[1])
        if domain_vals else ("none", 0)
    )
    branch = f"2B — targeted fix on weak bucket: {weakest_bucket}"
elif pairwise >= 50 or hallucination_regression:
    branch = "2A — train more (MelcotCR / dev-mix / hard-case retrain)"
else:
    branch = "Phase 1 review — v4 may need fundamental rework, not Phase 2"

print(f"Recommended Phase 2 branch: {branch}")
print()

# CoRPO viability check (orthogonal to A/B/C)
# Viable if we have a real correctness signal — both IoU + hit-rate non-trivial
if (iou_lenient >= 30 or hit_lenient >= 25) and pairwise >= 50:
    print("CoRPO (Phase 2D) is viable — verifiable correctness signal exists")
elif pairwise < 50:
    print("CoRPO (Phase 2D) deferred — v4 needs to beat base first before RL")
else:
    print("CoRPO (Phase 2D) deferred — IoU/hit-rate too low; reward signal would be noise")